# Customer Churn — Data Quality and Exploratory Analysis

**Objective:** Audit and clean the supplied customer churn dataset, document quality decisions, and inspect initial patterns.

> The supplied file is a small synthetic sample. Interpret patterns as exploratory only.

## 1. Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: seaborn is only used for a convenient chart style.
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

df = pd.read_csv("customer_churn_sample.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())

## 2. Profile columns, types, missing values, and duplicates

In [ ]:
print("Rows and columns:", df.shape)
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}))
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())
display(df.describe(include="all").T)

## 3. Clean and validate
Text values are trimmed. No rows are removed automatically: duplicates, missing values, and validation failures are checked first. Outliers are flagged for review rather than deleted solely because they exceed an IQR threshold.

In [ ]:
clean = df.copy()

# Normalize whitespace in text columns.
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype("string").str.strip()

expected_categories = {
    "Gender": {"Female", "Male"},
    "SubscriptionType": {"Basic", "Pro", "Enterprise"},
    "ContractType": {"Month-to-Month", "One Year", "Two Year"},
    "PaymentMethod": {"Credit Card", "Bank Transfer", "UPI", "Debit Card"},
    "Churn": {"Yes", "No"},
}
category_issues = {}
for col, allowed in expected_categories.items():
    observed = set(clean[col].dropna().astype(str))
    category_issues[col] = sorted(observed - allowed)

checks = {
    "Missing cells": int(clean.isna().sum().sum()),
    "Exact duplicate rows": int(clean.duplicated().sum()),
    "Duplicate CustomerID values": int(clean["CustomerID"].duplicated().sum()),
    "Age outside 18–100": int((~clean["Age"].between(18, 100)).sum()),
    "Negative tenure": int((clean["TenureMonths"] < 0).sum()),
    "Non-positive monthly charges": int((clean["MonthlyCharges"] <= 0).sum()),
    "Negative total charges": int((clean["TotalCharges"] < 0).sum()),
    "Negative support tickets": int((clean["SupportTickets"] < 0).sum()),
    "TotalCharges mismatch": int((clean["TotalCharges"].round(2) !=
                                   (clean["MonthlyCharges"] * clean["TenureMonths"]).round(2)).sum()),
    "Invalid category labels": int(sum(len(v) for v in category_issues.values())),
}
display(pd.Series(checks, name="Issue count").to_frame())
print("Invalid categories by column:", category_issues)

# No rows are dropped unless investigation proves they are duplicates or invalid.
# For this supplied sample, validation results should be reviewed before any changes.
clean.to_csv("cleaned_customer_churn.csv", index=False)
print("Saved cleaned_customer_churn.csv", clean.shape)

## 4. Outlier screening (IQR rule)
The 1.5×IQR rule identifies candidates for review; it does not prove the values are wrong. Do not remove them without a domain-based reason.

In [ ]:
numeric_cols = ["Age", "TenureMonths", "MonthlyCharges", "TotalCharges", "SupportTickets"]
outlier_summary = []
for col in numeric_cols:
    q1 = clean[col].quantile(0.25)
    q3 = clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = clean.loc[(clean[col] < lower) | (clean[col] > upper), col].tolist()
    outlier_summary.append({
        "column": col, "Q1": q1, "Q3": q3,
        "lower_bound": lower, "upper_bound": upper,
        "flagged_count": len(flagged), "flagged_values": flagged
    })
display(pd.DataFrame(outlier_summary))

## 5. Summary statistics and distributions

In [ ]:
display(clean[numeric_cols].describe().T)
display(clean["Churn"].value_counts().rename_axis("Churn").to_frame("Customers"))
print(f"Observed churn rate: {(clean['Churn'].eq('Yes').mean() * 100):.1f}%")

# Numeric distributions
clean[numeric_cols].hist(figsize=(12, 8), bins=8, edgecolor="black")
plt.suptitle("Numeric feature distributions")
plt.tight_layout()
plt.show()

# Churn count
churn_counts = clean["Churn"].value_counts()
churn_counts.plot(kind="bar", rot=0)
plt.title("Customer churn counts")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## 6. Explore churn by contract, subscription, and support tickets

In [ ]:
clean["ChurnFlag"] = clean["Churn"].eq("Yes").astype(int)

contract_summary = clean.groupby("ContractType").agg(
    customers=("CustomerID", "count"),
    churned=("ChurnFlag", "sum"),
    churn_rate=("ChurnFlag", "mean")
)
contract_summary["churn_rate"] *= 100
display(contract_summary.sort_values("churn_rate", ascending=False))

subscription_summary = clean.groupby("SubscriptionType").agg(
    customers=("CustomerID", "count"),
    churned=("ChurnFlag", "sum"),
    churn_rate=("ChurnFlag", "mean")
)
subscription_summary["churn_rate"] *= 100
display(subscription_summary.sort_values("churn_rate", ascending=False))

support_summary = clean.groupby("Churn")["SupportTickets"].agg(["count", "mean", "median"])
display(support_summary)

contract_summary["churn_rate"].plot(kind="bar", rot=20)
plt.title("Churn rate by contract type")
plt.xlabel("Contract type")
plt.ylabel("Churn rate (%)")
plt.tight_layout()
plt.show()

subscription_summary["churn_rate"].plot(kind="bar", rot=0)
plt.title("Churn rate by subscription type")
plt.xlabel("Subscription type")
plt.ylabel("Churn rate (%)")
plt.tight_layout()
plt.show()

clean.drop(columns=["ChurnFlag"]).to_csv("cleaned_customer_churn.csv", index=False)

## 7. Data dictionary and conclusions

### Data dictionary
| Column | Meaning |
|---|---|
| `CustomerID` | Unique customer identifier |
| `Gender` | Customer gender category |
| `Age` | Age in years |
| `TenureMonths` | Subscription tenure in months |
| `SubscriptionType` | Subscription tier |
| `MonthlyCharges` | Monthly charge |
| `TotalCharges` | Total charges recorded |
| `ContractType` | Contract duration |
| `SupportTickets` | Number of support tickets |
| `PaymentMethod` | Payment method |
| `Churn` | Whether the customer churned (`Yes`/`No`) |

### Conclusion template
Use the actual output above to describe:
1. Whether missing values, duplicates, invalid categories, or range violations were found.
2. Which variables have the widest distributions or potential outlier candidates.
3. Which contract/subscription groups show higher observed churn in this sample.
4. Why the sample size limits confidence and generalization.

**Important:** Describe associations, not causation. This is only 15 synthetic records.